In [1]:
#pip install polars
#pip install ml4t-engineer

In [2]:
import polars as pl
from ml4t.engineer import compute_features
import pandas as pd

c:\Users\ideac\AppData\Local\Programs\Python\Python313\Lib\site-packages\ml4t\engineer\features\ml\__init__.py:9: UserWarning: Feature 'cyclical_encode': lookback=0 but has period/window parameter. Consider using lookback='period' or specifying the actual lookback.
  from ml4t.engineer.features.ml.cyclical_encode import *  # noqa: F403


In [3]:
prices = pl.read_csv("../cleaned_prices.csv")
macro = pl.read_csv("../cleaned_macro.csv")

#Note: Previously, the columns in prices and macro did not have a "date" label, and tickers was undefined.
tickers = prices.columns[1:]
prices = prices.with_columns(pl.col(prices.columns[0]).cast(pl.Date).alias("date")).drop(prices.columns[0])
macro  = macro.with_columns(pl.col(macro.columns[0]).cast(pl.Date).alias("date")).drop(macro.columns[0])

In [4]:
feature_list = ["rsi", "macd", "bollinger_bands", "sma"]

all_ticker_features = []

for ticker in tickers:
    df_ticker = prices.select([
        pl.col("date"),
        pl.col(ticker).cast(pl.Float64).alias("close")
    ]).with_columns([
        pl.col("close").alias("high"),
        pl.col("close").alias("low"),
        pl.col("close").alias("open"),
        pl.lit(1.0).alias("volume")
    ])

    features = compute_features(df_ticker, feature_list)

    # 20-month SMA distance (existing)
    features = features.with_columns(
        ((pl.col("close") / pl.col("sma")) - 1).alias("dist_from_sma")
    )

    # 10-month SMA — Faber's signal period
    features = features.with_columns(
        pl.col("close").rolling_mean(window_size=10).alias("sma10")
    )
    features = features.with_columns([
        ((pl.col("close") / pl.col("sma10")) - 1).alias("dist_from_10m_sma"),
        (pl.col("close") > pl.col("sma10")).cast(pl.Int8).alias("above_10m_sma"),
    ]).drop("sma10")

    features = features.rename({
        col: f"{ticker}_{col}" for col in features.columns if col != "date"
    })

    all_ticker_features.append(features)

final_df = all_ticker_features[0]
for next_df in all_ticker_features[1:]:
    final_df = final_df.join(next_df, on="date", how="left")

final_df = final_df.join(macro, on="date", how="left")
final_df = final_df.drop_nulls()

print(final_df.head())
print(f"\nShape: {final_df.shape}")
# Verify new columns are present
new_cols = [c for c in final_df.columns if "10m" in c]
print(f"New 10m SMA columns: {new_cols}")

shape: (5, 66)
┌────────────┬────────────┬────────────┬────────────┬───┬──────────┬────────┬──────────┬───────┐
│ date       ┆ SPY_close  ┆ SPY_high   ┆ SPY_low    ┆ … ┆ CPIAUCSL ┆ T10Y2Y ┆ INDPRO   ┆ TB3MS │
│ ---        ┆ ---        ┆ ---        ┆ ---        ┆   ┆ ---      ┆ ---    ┆ ---      ┆ ---   │
│ date       ┆ f64        ┆ f64        ┆ f64        ┆   ┆ f64      ┆ f64    ┆ f64      ┆ f64   │
╞════════════╪════════════╪════════════╪════════════╪═══╪══════════╪════════╪══════════╪═══════╡
│ 2007-09-30 ┆ 108.383331 ┆ 108.383331 ┆ 108.383331 ┆ … ┆ 208.292  ┆ 0.62   ┆ 114.357  ┆ 3.89  │
│ 2007-10-31 ┆ 109.853653 ┆ 109.853653 ┆ 109.853653 ┆ … ┆ 208.903  ┆ 0.54   ┆ 113.9957 ┆ 3.9   │
│ 2007-11-30 ┆ 105.598846 ┆ 105.598846 ┆ 105.598846 ┆ … ┆ 210.565  ┆ 0.93   ┆ 113.9019 ┆ 3.27  │
│ 2007-12-31 ┆ 104.409729 ┆ 104.409729 ┆ 104.409729 ┆ … ┆ 211.16   ┆ 0.99   ┆ 113.9673 ┆ 3.0   │
│ 2008-01-31 ┆ 98.096962  ┆ 98.096962  ┆ 98.096962  ┆ … ┆ 212.516  ┆ 1.5    ┆ 114.2334 ┆ 2.75  │
└────────────┴─

In [5]:
final_df

date,SPY_close,SPY_high,SPY_low,SPY_open,SPY_volume,SPY_bollinger_bands,SPY_macd,SPY_rsi,SPY_sma,SPY_dist_from_sma,SPY_dist_from_10m_sma,SPY_above_10m_sma,EFA_close,EFA_high,EFA_low,EFA_open,EFA_volume,EFA_bollinger_bands,EFA_macd,EFA_rsi,EFA_sma,EFA_dist_from_sma,EFA_dist_from_10m_sma,EFA_above_10m_sma,IEF_close,IEF_high,IEF_low,IEF_open,IEF_volume,IEF_bollinger_bands,IEF_macd,IEF_rsi,IEF_sma,IEF_dist_from_sma,IEF_dist_from_10m_sma,IEF_above_10m_sma,VNQ_close,VNQ_high,VNQ_low,VNQ_open,VNQ_volume,VNQ_bollinger_bands,VNQ_macd,VNQ_rsi,VNQ_sma,VNQ_dist_from_sma,VNQ_dist_from_10m_sma,VNQ_above_10m_sma,DBC_close,DBC_high,DBC_low,DBC_open,DBC_volume,DBC_bollinger_bands,DBC_macd,DBC_rsi,DBC_sma,DBC_dist_from_sma,DBC_dist_from_10m_sma,DBC_above_10m_sma,FEDFUNDS,CPIAUCSL,T10Y2Y,INDPRO,TB3MS
date,f64,f64,f64,f64,f64,struct[3],f64,f64,f64,f64,f64,i8,f64,f64,f64,f64,f64,struct[3],f64,f64,f64,f64,f64,i8,f64,f64,f64,f64,f64,struct[3],f64,f64,f64,f64,f64,i8,f64,f64,f64,f64,f64,struct[3],f64,f64,f64,f64,f64,i8,f64,f64,f64,f64,f64,struct[3],f64,f64,f64,f64,f64,i8,f64,f64,f64,f64,f64
2007-09-30,108.383331,108.383331,108.383331,108.383331,1.0,"{111.091646,97.266312,83.440979}",NaN,75.091516,97.266312,0.114295,0.048917,1,46.893772,46.893772,46.893772,46.893772,1.0,"{48.426662,40.742857,33.059051}",NaN,82.834235,40.742857,0.150969,0.060478,1,52.976429,52.976429,52.976429,52.976429,1.0,"{53.340062,50.256049,47.172036}",NaN,72.067073,50.256049,0.05413,0.030466,1,32.989185,32.989185,32.989185,32.989185,1.0,"{38.465243,32.31626,26.167276}",NaN,59.080648,32.31626,0.020823,-0.037596,0,22.830553,22.830553,22.830553,22.830553,1.0,"{22.121982,20.241359,18.360735}",NaN,72.026673,20.241359,0.127916,0.098347,1,4.94,208.292,0.62,114.357,3.89
2007-10-31,109.853653,109.853653,109.853653,109.853653,1.0,"{112.558329,98.345076,84.131823}",NaN,76.35366,98.345076,0.117022,0.052398,1,48.886707,48.886707,48.886707,48.886707,1.0,"{49.389793,41.451771,33.513749}",NaN,85.1725,41.451771,0.179364,0.087561,1,53.547035,53.547035,53.547035,53.547035,1.0,"{53.81745,50.490169,47.162888}",NaN,74.20992,50.490169,0.060544,0.035431,1,33.681644,33.681644,33.681644,33.681644,1.0,"{38.439854,32.601694,26.763535}",NaN,60.603152,32.601694,0.033126,-0.014908,0,24.771679,24.771679,24.771679,24.771679,1.0,"{23.128055,20.556759,17.985463}",NaN,77.712999,20.556759,0.205038,0.164789,1,4.76,208.903,0.54,113.9957,3.9
2007-11-30,105.598846,105.598846,105.598846,105.598846,1.0,"{113.109575,99.138253,85.166932}",NaN,65.940843,99.138253,0.065168,0.007,1,47.115196,47.115196,47.115196,47.115196,1.0,"{49.907159,42.002604,34.098048}",NaN,75.347454,42.002604,0.121721,0.036706,1,55.714355,55.714355,55.714355,55.714355,1.0,"{54.728051,50.865834,47.003617}",NaN,80.369784,50.865834,0.09532,0.066175,1,30.491686,30.491686,30.491686,30.491686,1.0,"{38.386121,32.661684,26.937246}",NaN,51.159656,32.661684,-0.066439,-0.089327,0,24.617363,24.617363,24.617363,24.617363,1.0,"{23.854217,20.838678,17.82314}",NaN,76.383678,20.838678,0.18133,0.130253,1,4.49,210.565,0.93,113.9019,3.27
2007-12-31,104.409729,104.409729,104.409729,104.409729,1.0,"{113.425781,99.815299,86.204817}",NaN,63.340967,99.815299,0.046029,-0.009622,0,45.710571,45.710571,45.710571,45.710571,1.0,"{50.214969,42.39674,34.578511}",NaN,68.591218,42.39674,0.078162,-0.002125,0,55.742569,55.742569,55.742569,55.742569,1.0,"{55.417027,51.258023,47.099018}",NaN,80.435294,51.258023,0.08749,0.057788,1,28.841177,28.841177,28.841177,28.841177,1.0,"{38.334579,32.68896,27.043341}",NaN,47.07249,32.68896,-0.117709,-0.11828,0,26.267769,26.267769,26.267769,26.267769,1.0,"{24.954627,21.136525,17.318423}",NaN,80.270667,21.136525,0.242767,0.174784,1,4.24,211.16,0.99,113.9673,3.0
2008-01-31,98.096962,98.096962,98.096962,98.096962,1.0,"{112.865121,100.313559,87.761997}",NaN,51.689509,100.313559,-0.022097,-0.067854,0,42.123611,42.123611,42.123611,42.123611,1.0,"{50.003299,42.683814,35.36433}",NaN,55.022785,42.683814,-0.013124,-0.078017,0,57.613224,57.613224,57.613224,57.613224,1

In [6]:
#DataFrame was using the polars module. I converted it into a pandas DataFrame because csv does not work well with nested information.
#Resulting DF may end up looking weird but it should still be usable since the data can be extracted properly regardless
final_df_pandas = final_df.to_pandas()

In [7]:
final_df_pandas

,date,SPY_close,SPY_high,SPY_low,SPY_open,SPY_volume,SPY_bollinger_bands,SPY_macd,SPY_rsi,SPY_sma,...,DBC_rsi,DBC_sma,DBC_dist_from_sma,DBC_dist_from_10m_sma,DBC_above_10m_sma,FEDFUNDS,CPIAUCSL,T10Y2Y,INDPRO,TB3MS
0,2007-09-30,108.383331,108.383331,108.383331,108.383331,1.0,"{'upper': 111.09164557911396, 'middle': 97.266...",NaN,75.091516,97.266312,...,72.026673,20.241359,0.127916,0.098347,1,4.94,208.292,0.62,114.3570,3.89
1,2007-10-31,109.853653,109.853653,109.853653,109.853653,1.0,"{'upper': 112.55832900427468, 'middle': 98.345...",NaN,76.353660,98.345076,...,77.712999,20.556759,0.205038,0.164789,1,4.76,208.903,0.54,113.9957,3.90
2,2007-11-30,105.598846,105.598846,105.598846,105.598846,1.0,"{'upper': 113.10957481188046, 'middle': 99.138...",NaN,65.940843,99.138253,...,76.383678,20.838678,0.181330,0.130253,1,4.49,210.565,0.93,113.9019,3.27
3,2007-12-31,104.409729,104.409729,104.409729,104.409729,1.0,"{'upper': 113.42578115508756, 'middle': 99.815...",NaN,63.340967,99.815299,...,80.270667,21.136525,0.242767,0.174784,1,4.24,211.160,0.99,113.9673,3.00
4,2008-01-31,98.096962,98.096962,98.096962,98.096962,1.0,"{'upper': 112.86512141749013, 'middle': 100.31...",NaN,51.689509,100.313559,...,81.769867,21.476622,0.258741,0.175284,1,3.94,212.516,1.50,114.2334,2.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202,2024-07-31,539.458191,539.458191,539.458191,539.458191,1.0,"{'upper': 549.7205806756684, 'middle': 445.743...",38.022125,70.905991,445.743411,...,51.482578,20.950806,-0.010173,-0.009224,0,5.33,313.534,-0.20,102.8887,5.20
203,2024-08-31,552.062988,552.062988,552.062988,552.062988,1.0,"{'upper': 562.1412634687633, 'middle': 455.015...",40.537824,72.445038,455.015788,...,48.648786,20.887187,-0.027824,-0.024488,0,5.33,314.121,0.00,103.1389,5.05
204,2024-09-30,563.658813,563.658813,563.658813,563.658813,1.0,"{'upper': 576.3198956064132, 'middle': 463.715...",42.971866,73.817228,463.715181,...,49.664544,20.821286,-0.017690,-0.015168,0,5.13,314.686,0.15,102.6418,4.72
205,2024-10-31,558.628967,558.628967,558.628967,558.628967,1.0,"{'upper': 585.5959849118509, 'middle': 472.652...",43.987930,72.139108,472.652951,...,51.721409,20.818663,-0.003449,-0.003441,0,4.83,315.454,0.12,102.2805,4.51


In [8]:
final_df_pandas.to_csv('../feature_engineer_csv.csv', index=False)